# TotalSpineSeg FULL mode on Duke T2 — per-vertebra labels for Group 5.2 (fracture)

Produces **`step2_output`** (canonical per-vertebra labels: C1=11 … C7=17, T1=21…, cord=1, canal=2, discs 63–100),
which `run_fracture_on_tss.py` consumes locally.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. Put **`fracture_batch.zip`** in your Google Drive (My Drive root).

Then Runtime → **Run all**. Approve the Drive popup.

Watch the **`✓ Ns`** timer: **~1–3 min/case = GPU**; **~20–40 min = CPU** (stop, fix GPU).
Unlike the SCIseg notebook, **TotalSpineSeg uses Colab's native CUDA torch directly — no torch-swap needed.**

### 1 — confirm GPU is attached

In [ ]:
!nvidia-smi -L   # must list a GPU; if blank, Runtime -> T4 GPU and re-run

### 2 — install TotalSpineSeg (~3–5 min)
`totalspineseg` does **not** pull `nnunetv2`; pin **`kornia<0.8`** (0.8 removed `kornia.core.Tensor` → auglab crash).

In [ ]:
!pip install -q totalspineseg nnunetv2
!pip install -q "kornia<0.8"   # downgrade if totalspineseg pulled 0.8.x
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

### 3 — mount Google Drive (approve the popup)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 4 — unzip the 3 scans from Drive

In [ ]:
import os, glob
!unzip -q -o /content/drive/MyDrive/fracture_batch.zip -d /content/cases
os.makedirs('/content/drive/MyDrive/fracture_masks', exist_ok=True)
inputs = sorted(glob.glob('/content/cases/*.nii.gz'))
print(len(inputs), 'input scans:', [os.path.basename(f) for f in inputs])

### 5 — run TSS in FULL mode (NO `--step1`); save `step2_output` to Drive (resumable)
First case downloads ~2 GB weights (one-time). A Colab restart wipes weights → re-run from cell 2.

In [ ]:
import os, glob, shutil, time, subprocess
OUT = '/content/drive/MyDrive/fracture_masks'; os.makedirs(OUT, exist_ok=True)
inputs = sorted(glob.glob('/content/cases/*.nii.gz'))
print(len(inputs), 'cases\n')

for i, f in enumerate(inputs, 1):
    base = os.path.basename(f)[:-7]
    dst = f'{OUT}/{base}_step2.nii.gz'
    if os.path.exists(dst):
        print(f'[{i}/{len(inputs)}] skip (done): {base}'); continue
    outdir = f'/content/out/{base}'
    t = time.time(); print(f'[{i}/{len(inputs)}] {base} ...', flush=True)
    # FULL mode = no --step1 -> step2_output has per-vertebra canonical labels
    r = subprocess.run(['totalspineseg', f, outdir, '--device', 'cuda'],
                       capture_output=True, text=True)
    seg = glob.glob(f'{outdir}/step2_output/*.nii.gz')
    if r.returncode == 0 and seg:
        shutil.copy(seg[0], dst)
        print(f'    ✓ {time.time()-t:.0f}s -> {os.path.basename(dst)}')
    else:
        print('    ERROR:', (r.stderr or r.stdout)[-800:])
print('\nDONE. step2 masks on Drive:', len(glob.glob(f'{OUT}/*_step2.nii.gz')))

### Done
Masks are in **My Drive/fracture_masks/** (`*_step2.nii.gz`). Download them into `~/dev/group5-proto/out/` and run locally:
```bash
python run_fracture_on_tss.py out/case1_uid3_step2.nii.gz out/case2_uid20_step2.nii.gz out/case3_uid37_step2.nii.gz
```
**If it disconnects:** re-run cells **2 → 5** (install + weights re-download, loop skips done cases).